# ⚖️ Legal Act Classification — EDA

สมุดนี้มีไว้ **สำรวจข้อมูลอย่างเดียว** logic ทั้งหมดอยู่ใน `src/legal_act/`

จุดประสงค์คือยืนยันข้อสังเกต 3 ข้อที่กำหนดวิธีทำทั้งโปรเจกต์ (ดูรายละเอียดใน README)

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

import pandas as pd
from legal_act.config import load_config
from legal_act.data import clause_table, load_committees, load_patterns, load_split
from legal_act.utils import normalise_name

cfg = load_config(pathlib.Path.cwd().parent / "configs/default.yaml")
train = load_split(cfg, "train")
test = load_split(cfg, "test")
committees = load_committees(cfg)
patterns = load_patterns(cfg)
train.head(3)

## 1. label เป็นฟังก์ชัน deterministic

ถ้า `(context, เซ็ตผู้ลงนาม, legal_act)` เดียวกันให้คำตอบเดียวกันเสมอ แปลว่าไม่มี noise ในเฉลย —
เพดานของงานนี้คือ 1.00 และทุกข้อที่ผิดคือ "เราอ่านกฎผิด" ไม่ใช่ "โมเดลยังไม่ converge"

In [ ]:
key = train.apply(lambda r: (r["context"], tuple(sorted(r["signers"])), r["legal_act"]), axis=1)
g = train.assign(k=key).groupby("k")["answer"].nunique()
print(f"กลุ่มคำถามที่ไม่ซ้ำ : {len(g):,}")
print(f"กลุ่มที่ label ขัดกันเอง : {(g > 1).sum()}")
print(f"\nสัดส่วนคำตอบ:\n{train['answer'].value_counts(normalize=True).round(4)}")

## 2. `context` ซ้ำกันมหาศาล

นี่คือเหตุผลที่เรายิง LLM ต่อ *clause* ไม่ใช่ต่อ *แถว*

In [ ]:
clauses = clause_table([train, test])
rows = len(train) + len(test)
print(f"แถวทั้งหมด        : {rows:,}")
print(f"clause ที่ไม่ซ้ำ   : {len(clauses):,}   ({rows / len(clauses):.0f}x fewer llm calls)")
clauses[["clause_id", "pattern", "n_rows", "splits"]].head(10)

In [ ]:
# clause ไม่กี่อันกินแถวไปครึ่งหนึ่ง — คอมไพล์อันบนสุดให้ถูกคือกำไรมหาศาล
cum = clauses["n_rows"].cumsum() / clauses["n_rows"].sum()
print(f"clause 10 อันแรกครอบคลุม {cum.iloc[9]:.1%} ของแถวทั้งหมด")
print(f"clause 50 อันแรกครอบคลุม {cum.iloc[49]:.1%} ของแถวทั้งหมด")

## 3. ชื่อผู้ลงนามเป็นกรรมการจริงเกือบ 100%

ทำให้เรา "snap" ชื่อที่ LLM ถอดออกมาจากข้อความ กลับไปหาชื่อกรรมการจริงได้อย่างปลอดภัย

In [ ]:
hits = misses = 0
bad = []
for df in (train, test):
    for rg, signers in zip(df["rg"], df["signers"]):
        pool = committees[rg].by_key if rg in committees else {}
        for n in signers:
            if normalise_name(n) in pool:
                hits += 1
            else:
                misses += 1
                bad.append((rg, n))
print(f"ชื่อที่ตรงกับ committee.csv : {hits:,} / {hits + misses:,}")
print(f"ที่ไม่ตรง                  : {bad}")

## 4. หน้าตาของ pattern ทั้ง 62 แบบ

ทุกแบบยุบเป็น "slot + จำนวนที่ต้องเป๊ะ" ได้หมด ซึ่งเป็นที่มาของ DSL ใน `src/legal_act/rules.py`

In [ ]:
both = pd.concat([train, test], ignore_index=True)
print(pd.crosstab(both["pattern"].astype(str).str[:2], both["split"]))
print()
for p in sorted(train["pattern"].unique())[:8]:
    print(f"{p}  {patterns.get(int(p), '?')}")

In [ ]:
# ตัวอย่าง clause ที่มีข้อยกเว้น — จุดที่ยากที่สุดของโจทย์
sub = train[train["condition"] != ""]
r = sub.iloc[0]
print("CLAUSE   :", r["context"][:400])
print("\nCONDITION:", r["condition"][:300])
print("\nแถวของ clause นี้:")
sub[sub["clause_id"] == r["clause_id"]][["legal_act", "signers", "answer"]].head(8)